# MediCS — Colab GPU Runner

Run cells in order. Each cell is independent — re-run any cell to check scores.

**Pipeline:** Setup → Dataset → Attack-Defense Loop (3 rounds) → DPO → Eval → Transfer → Detection → Figures → Upload

In [2]:
# Cell 1 — Setup (run once per session)
from google.colab import drive
drive.mount('/content/drive')

import sys, os
os.chdir('/content/drive/MyDrive/MediCS')
sys.path.insert(0, '/content/drive/MyDrive/MediCS')

%pip install -q -r requirements.txt
!nvidia-smi
print(f"\nWorking dir: {os.getcwd()}")
print("Setup complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thu Apr  2 04:10:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             52W /  400W |       0MiB /  81920MiB |      0%      Default |
|          

## Phase A — Dataset Construction (one-time, no GPU needed)

In [ ]:
# Cell 2 — Build MediCS-500 dataset (seeds → keywords → translation → splits)
# NOTE: Calls OpenAI API for keyword extraction — costs ~$1-2
!python scripts/01_build_dataset.py --config config/experiment_config.yaml

[TIMING] Starting: Dataset Construction
API keys loaded from .env file.
=== MediCS-500 Dataset Builder ===
Languages: ['hi', 'bn', 'sw', 'yo', 'tl', 'gu']
Semantic threshold: 0.75

--- Step 1-2: Loading and deduplicating seeds ---
Loaded 500 raw seeds
Deduplication: 500 -> 500 (removed 0 duplicates)
After dedup: 500 seeds → saved to data/seeds/deduped_seeds.jsonl

--- Step 3: Extracting keywords ---
  Resuming from checkpoint: 500 keywords already extracted
  All 500 keywords already extracted (from checkpoint)
Extracted keywords for 500 seeds

--- Step 4: Code-switching ---
Code-switching: 100% 3000/3000 [00:03<00:00, 849.69it/s] 
Generated 3000 code-switched variants

=== Semantic Preservation Verification ===
modules.json: 100% 349/349 [00:00<00:00, 1.31MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 437kB/s]
README.md: 10.5kB [00:00, 4.96MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 277kB/s]
config.json: 100% 612/612 [00:00<00:00, 3.35MB/s]
model

In [ ]:
# Cell 3 — Token fragmentation analysis (no GPU, no API cost)
!python scripts/07_tokenization_analysis.py

[TIMING] Starting: Tokenization Analysis
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Loaded 2665 seeds from data/medics_500/medics_500_full.jsonl
Loaded keywords for 500 seeds

Analyzing tokenization for meta-llama/Meta-Llama-3-8B-Instruct
Languages: ['hi', 'bn', 'sw', 'yo', 'tl', 'gu']
  Tokenization analysis: 50/2665 seeds processed
  Tokenization analysis: 100/2665 seeds processed
  Tokenization analysis: 150/2665 seeds processed
  Tokenization analysis: 200/2665 seeds processed
  Tokenization analysis: 250/2665 seeds processed
  Tokenization analysis: 300/2665 seeds processed
  Tokenization analysis: 350/2665 seeds processed
  Tokenization analysis: 400/2665 seeds processed
  Tokenization analysis: 450/2665 seeds processed
  Tokenization analysis: 500/2665 seeds processed
  Tokenization analysis: 550/2665 seeds processed
  Tokenization analysis: 600/2665 seeds processed
  Tokenization analysis: 650

## Round 1 — Base Model Attack

In [ ]:
# Cell 4 — Round 1: Generate attack prompts (Thompson Sampling)
!python scripts/02_run_attack_round.py --round 1 --phase generate --config config/experiment_config.yaml

[TIMING] Starting: Attack Round
Generation attempt 1/3
Category quotas (strict): {'TOX': 34, 'SH': 33, 'MIS': 34, 'ULP': 33, 'PPV': 33, 'UCA': 33}
  [25/200] generated
  [50/200] generated
  [75/200] generated
  [100/200] generated
  [125/200] generated
  [150/200] generated
  [175/200] generated
  [200/200] generated
MTE generation summary: 0/29 API fallbacks (0.0%)
Quality summary: duplicates=0, category_span=1, mte_fallback=0.0% (0/35), mte_min_turns=3, cs_obf_leet_mean=0.0775

Generated 200 attack prompts → results/attacks/round_1/prompts.jsonl
Strategy distribution (generated): {'CS-OBF': 47, 'CS': 41, 'RP': 33, 'MTE': 35, 'CS-RP': 44}
Category distribution (generated): {'SH': 33, 'PPV': 33, 'TOX': 34, 'ULP': 33, 'MIS': 34, 'UCA': 33}
Next: run colab/run_inference.py on Colab, then come back with --phase judge
[TIMING] Finished: Attack Round (76.4s / 1.3min)
[TIMING] Report saved to results/timing_report.json (7 entries)


In [ ]:
# Cell 5 — Round 1: Inference (base model, BASE prompt)
!python colab/run_inference.py \
    --checkpoint base --system-prompt base --seed 42 \
    --input results/attacks/round_1/prompts.jsonl \
    --output results/attacks/round_1/responses.jsonl

[TIMING] Starting: Inference
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
=== Target Model Inference ===
Model: meta-llama/Meta-Llama-3-8B-Instruct
Checkpoint: base
System prompt: BASE (no safety priming) [override]
Seed: 42
Input: results/attacks/round_1/prompts.jsonl
config.json: 100% 654/654 [00:00<00:00, 2.91MB/s]
model.safetensors.index.json: 23.9kB [00:00, 15.7MB/s]
Fetching 4 files: 100% 4/4 [00:38<00:00,  9.70s/it]
Download complete: 100% 16.1G/16.1G [00:38<00:00, 413MB/s]                
Loading weights: 100% 291/291 [00:04<00:00, 62.05it/s, Materializing param=model.norm.weight]                               
generation_config.json: 100% 187/187 [00:00<00:00, 925kB/s]
tokenizer_config.json: 51.0kB [00:00, 66.9MB/s]
tokenizer.json: 9.09MB [00:01, 7.41MB/s]
special_tokens_map.json: 100% 73.0/73.0 [00:00<00:00, 348kB/s]
Loaded 200 prompts
Setting `pad_token_id` to `eos_token_id`:128001 for open-e

In [ ]:
# Cell 6 — Round 1: Judge responses (GPT-5)
# NOTE: This calls the OpenAI API — costs ~$0.40
!python scripts/02_run_attack_round.py --round 1 --phase judge --config config/experiment_config.yaml


[TIMING] Starting: Attack Round
Judge model: gpt-5-chat
Judging responses: 100% 200/200 [03:38<00:00,  1.09s/it]
Judge content-filter summary: hits=6, redacted_attempts=0, redacted_successes=0, redacted_failures=0, redacted_skipped=6

--- API Cost Summary ---
Total calls: 195
  gpt-5-chat: 195 calls, 143206 in / 9592 out, ~$0.2749
Estimated session cost: $0.2749
Judge error rate: 0.0%
Judge fallback rate: 3.0% (6/200 used heuristic)

Round 1 ASR: 56.5% (113/200)
Updated bandit state: {'CS': 0.456, 'RP': 0.7920792079207921, 'MTE': 0.2616822429906542, 'CS-RP': 0.7313432835820896, 'CS-OBF': 0.4965034965034965}
[TIMING] Finished: Attack Round (220.2s / 3.7min)
[TIMING] Report saved to results/timing_report.json (11 entries)


In [ ]:
# Cell 7 — Round 1: Check ASR
from medics.metrics import compute_asr, compute_per_strategy_asr, compute_per_category_asr
from medics.utils import load_jsonl

results = load_jsonl("results/attacks/round_1/results.jsonl")
print(f"Round 1 ASR: {compute_asr(results):.1%} ({sum(1 for r in results if r.get('judge_label')=='harmful')}/{len(results)})")
print("\nBy strategy:")
for s, asr in sorted(compute_per_strategy_asr(results).items()):
    print(f"  {s}: {asr:.1%}")
print("\nBy category:")
for c, asr in sorted(compute_per_category_asr(results).items()):
    print(f"  {c}: {asr:.1%}")

Round 1 ASR: 56.5% (113/200)

By strategy:
  CS: 48.8%
  CS-OBF: 51.1%
  CS-RP: 77.3%
  MTE: 25.7%
  RP: 78.8%

By category:
  MIS: 38.2%
  PPV: 57.6%
  SH: 45.5%
  TOX: 67.6%
  UCA: 63.6%
  ULP: 66.7%


In [ ]:
# Cell 8 — Round 1: Build SFT data + Train SFT + Benign eval for DPO

  # Rebuild SFT data from cache ($0 API cost)
  !python scripts/03_build_defense_data.py --rounds 1 --type sft \
      --config config/experiment_config.yaml \
      --rebuild-from-cache --force

  # Train SFT round 1 (now with completion-only loss masking)
  !python colab/train_sft.py --round 1 --config config/experiment_config.yaml

  # Benign inference through SFT round 1 (needed for DPO over-refusal correction)
  !python colab/run_inference.py \
      --config config/experiment_config.yaml \
      --checkpoint checkpoints/sft/round_1/final --system-prompt base --seed 42 \
      --input data/seeds/benign_twins.jsonl \
        

[TIMING] Starting: Defense Data Construction
Round 1: 113 jailbreaks
Benign twins: 500
Cache: 278 refusals, 500 helpful responses
SFT data (from cache): 853 examples (113 refusals + 500 helpful + 180 prefix-recovery + 60 bare-RP-refusal)
SFT data saved: data/defense/sft_round_1.json (853 examples)
Refusal map saved: data/defense/refusals.json (278 entries: 113 prompt-specific, 165 legacy seed_id)
Helpful targets saved: data/defense/helpful_targets.json (500 entries)
[TIMING] Finished: Defense Data Construction (0.2s / 0.0min)
[TIMING] Report saved to results/timing_report.json (11 entries)
[TIMING] Starting: SFT Training
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
=== QLoRA-SFT Training: Round 1 ===
Model: meta-llama/Meta-Llama-3-8B-Instruct
4-bit compute dtype: bfloat16
Trainer precision: bf16=True fp16=False
Loading weights: 100% 291/291 [00:04<00:00, 63.88it/s, Materializing param=model.norm.weight]

## Round 2 — Attack SFT Round 1

In [ ]:
# Cell 9 — Round 2: Generate attacks
!python scripts/02_run_attack_round.py --round 2 --phase generate --config config/experiment_config.yaml

[TIMING] Starting: Attack Round
Generation attempt 1/3
Loading bandit state from round 1
Category quotas (strict): {'TOX': 33, 'SH': 33, 'MIS': 33, 'ULP': 34, 'PPV': 34, 'UCA': 33}
  [25/200] generated
  [50/200] generated
  [75/200] generated
  [100/200] generated
  [125/200] generated
  [150/200] generated
  [175/200] generated
  [200/200] generated
MTE generation summary: 0/34 API fallbacks (0.0%)
Quality summary: duplicates=0, category_span=1, mte_fallback=0.0% (0/34), mte_min_turns=3, cs_obf_leet_mean=0.0952

Generated 200 attack prompts → results/attacks/round_2/prompts.jsonl
Strategy distribution (generated): {'RP': 71, 'MTE': 34, 'CS': 16, 'CS-RP': 44, 'CS-OBF': 35}
Category distribution (generated): {'TOX': 33, 'SH': 33, 'ULP': 34, 'PPV': 34, 'MIS': 33, 'UCA': 33}
Next: run colab/run_inference.py on Colab, then come back with --phase judge
[TIMING] Finished: Attack Round (273.8s / 4.6min)
[TIMING] Report saved to results/timing_report.json (18 entries)


In [ ]:
# Cell 10 — Round 2: Inference (SFT round 1 checkpoint, BASE prompt)
!python colab/run_inference.py \
    --checkpoint checkpoints/sft/round_1/final --system-prompt base --seed 42 \
    --input results/attacks/round_2/prompts.jsonl \
    --output results/attacks/round_2/responses.jsonl

[TIMING] Starting: Inference
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
=== Target Model Inference ===
Model: meta-llama/Meta-Llama-3-8B-Instruct
Checkpoint: checkpoints/sft/round_1/final
System prompt: BASE (no safety priming) [override]
Seed: 42
Input: results/attacks/round_2/prompts.jsonl
config.json: 100% 654/654 [00:00<00:00, 2.76MB/s]
model.safetensors.index.json: 23.9kB [00:00, 47.0MB/s]
Fetching 4 files: 100% 4/4 [00:37<00:00,  9.35s/it]
Download complete: 100% 16.1G/16.1G [00:37<00:00, 429MB/s]                
Loading weights: 100% 291/291 [00:04<00:00, 65.17it/s, Materializing param=model.norm.weight]                               
generation_config.json: 100% 187/187 [00:00<00:00, 925kB/s]
tokenizer_config.json: 51.0kB [00:00, 64.5MB/s]
tokenizer.json: 9.09MB [00:01, 7.71MB/s]
special_tokens_map.json: 100% 73.0/73.0 [00:00<00:00, 367kB/s]
Loading adapter: checkpoints/sft/round_1/final
Loade

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 11 — Round 2: Judge + ASR
!python scripts/02_run_attack_round.py --round 2 --phase judge

from medics.metrics import compute_asr, compute_per_strategy_asr, compute_per_category_asr
from medics.utils import load_jsonl

results = load_jsonl("results/attacks/round_2/results.jsonl")
print(f"\nRound 2 ASR: {compute_asr(results):.1%} ({sum(1 for r in results if r.get('judge_label')=='harmful')}/{len(results)})")
print("\nBy strategy:")
for s, asr in sorted(compute_per_strategy_asr(results).items()):
    print(f"  {s}: {asr:.1%}")
print("\nBy category:")
for c, asr in sorted(compute_per_category_asr(results).items()):
    print(f"  {c}: {asr:.1%}")

[TIMING] Starting: Attack Round
Judge model: gpt-5-chat
Judging responses: 100% 200/200 [03:46<00:00,  1.13s/it]
Judge content-filter summary: hits=3, redacted_attempts=0, redacted_successes=0, redacted_failures=0, redacted_skipped=3

--- API Cost Summary ---
Total calls: 198
  gpt-5-chat: 198 calls, 107890 in / 9376 out, ~$0.2286
Estimated session cost: $0.2286
Judge error rate: 0.0%
Judge fallback rate: 3.0% (6/200 used heuristic)

Round 2 ASR: 55.5% (111/200)
Updated bandit state: {'CS': 0.4397163120567376, 'RP': 0.8197674418604651, 'MTE': 0.2695035460992908, 'CS-RP': 0.7415730337078652, 'CS-OBF': 0.4044943820224719}
[TIMING] Finished: Attack Round (229.0s / 3.8min)
[TIMING] Report saved to results/timing_report.json (21 entries)

Round 2 ASR: 55.5% (111/200)

By strategy:
  CS: 31.2%
  CS-OBF: 2.9%
  CS-RP: 77.3%
  MTE: 29.4%
  RP: 85.9%

By category:
  MIS: 6.1%
  PPV: 29.4%
  SH: 78.8%
  TOX: 81.8%
  UCA: 45.5%
  ULP: 91.2%


In [ ]:
# Cell 12 — Round 2: Build SFT data + Train SFT + Benign eval for DPO
!python scripts/03_build_defense_data.py --rounds 1,2 --type sft
!python colab/train_sft.py --round 2 --prev-checkpoint checkpoints/sft/round_1/final

# Benign inference through SFT round 2 (needed for DPO over-refusal correction)
!python colab/run_inference.py \
    --checkpoint checkpoints/sft/round_2/final --system-prompt base --seed 42 \
    --input data/seeds/benign_twins.jsonl \
    --output results/eval/sft/round_2/benign_results.jsonl

[TIMING] Starting: Defense Data Construction
Round 1: 113 jailbreaks
Round 2: 111 jailbreaks
Benign twins: 500
Generating refusals: 100% 224/224 [06:02<00:00,  1.62s/it]
Generating helpful responses: 100% 500/500 [45:55<00:00,  5.51s/it]  
SFT data: 448 examples (224 refusals + 224 helpful)
SFT data saved: data/defense/sft_round_1_2.json (448 examples)
Refusal map saved: data/defense/refusals.json (165 entries)
Helpful targets saved: data/defense/helpful_targets.json (224 entries)
[TIMING] Finished: Defense Data Construction (3122.4s / 52.0min)
[TIMING] Report saved to results/timing_report.json (22 entries)
[TIMING] Starting: SFT Training
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
=== QLoRA-SFT Training: Round 2 ===
Model: meta-llama/Meta-Llama-3-8B-Instruct
Loading weights: 100% 291/291 [00:04<00:00, 62.34it/s, Materializing param=model.norm.weight]                               
Loading previous ch

## Round 3 — Attack SFT Round 2

In [ ]:
# Cell 13 — Round 3: Generate attacks
!python scripts/02_run_attack_round.py --round 3 --phase generate --config config/experiment_config.yaml

[TIMING] Starting: Attack Round
Generation attempt 1/3
Loading bandit state from round 2
Category quotas (strict): {'TOX': 33, 'SH': 33, 'MIS': 34, 'ULP': 34, 'PPV': 33, 'UCA': 33}
  [25/200] generated
  [50/200] generated
  [75/200] generated
  [100/200] generated
  [125/200] generated
  [150/200] generated
  [175/200] generated
  [200/200] generated
Quality summary: duplicates=0, category_span=1, mte_fallback=0.0% (0/0), mte_min_turns=0, cs_obf_leet_mean=0.0000

Generated 200 attack prompts → results/attacks/round_3/prompts.jsonl
Strategy distribution (generated): {'RP': 108, 'CS-RP': 71, 'CS': 21}
Category distribution (generated): {'ULP': 34, 'PPV': 33, 'UCA': 33, 'SH': 33, 'MIS': 34, 'TOX': 33}
Next: run colab/run_inference.py on Colab, then come back with --phase judge
[TIMING] Finished: Attack Round (151.2s / 2.5min)
[TIMING] Report saved to results/timing_report.json (25 entries)


In [ ]:
# Cell 14 — Round 3: Inference (SFT round 2 checkpoint, BASE prompt)
!python colab/run_inference.py \
    --checkpoint checkpoints/sft/round_2/final --system-prompt base --seed 42 \
    --input results/attacks/round_3/prompts.jsonl \
    --output results/attacks/round_3/responses.jsonl

[TIMING] Starting: Inference
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
=== Target Model Inference ===
Model: meta-llama/Meta-Llama-3-8B-Instruct
Checkpoint: checkpoints/sft/round_2/final
System prompt: BASE (no safety priming) [override]
Seed: 42
Input: results/attacks/round_3/prompts.jsonl
Loading weights: 100% 291/291 [00:04<00:00, 62.55it/s, Materializing param=model.norm.weight]                               
Loading adapter: checkpoints/sft/round_2/final
Loaded 200 prompts
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
  [1/200] processed (35s elapsed, ~7044s remaining)
    checkpoint saved → results/attacks/round_3/responses.jsonl.partial
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
  [2/200] processed (70s elapsed, ~6940s remaining)
    checkpoint saved → results/attacks/round_3/responses.jsonl.partial
Setting `pad_token_id` to `eos_token_i

In [ ]:
# Cell 15 — Round 3: Judge + ASR
!python scripts/02_run_attack_round.py --round 3 --phase judge

from medics.metrics import compute_asr, compute_per_strategy_asr, compute_per_category_asr
from medics.utils import load_jsonl

results = load_jsonl("results/attacks/round_3/results.jsonl")
print(f"\nRound 3 ASR: {compute_asr(results):.1%} ({sum(1 for r in results if r.get('judge_label')=='harmful')}/{len(results)})")
print("\nBy strategy:")
for s, asr in sorted(compute_per_strategy_asr(results).items()):
    print(f"  {s}: {asr:.1%}")
print("\nBy category:")
for c, asr in sorted(compute_per_category_asr(results).items()):
    print(f"  {c}: {asr:.1%}")

In [ ]:
# Cell 16 — Final SFT (round 3) + Benign eval + DPO
!python scripts/03_build_defense_data.py --rounds 1,2,3 --type sft
!python colab/train_sft.py --round 3 --prev-checkpoint checkpoints/sft/round_2/final

# Benign inference through SFT round 3 (needed for DPO over-refusal correction)
!python colab/run_inference.py \
    --checkpoint checkpoints/sft/round_3/final --system-prompt base --seed 42 \
    --input data/seeds/benign_twins.jsonl \
    --output results/eval/sft/round_3/benign_results.jsonl

# Build DPO data (uses benign results from all 3 SFT rounds for over-refusal correction)
!python scripts/03_build_defense_data.py --rounds 1,2,3 --type dpo
!python colab/train_dpo.py --sft-checkpoint checkpoints/sft/round_3/final

## Held-Out Evaluation — 3 Checkpoints x 3 Seeds

In [ ]:
# Cell 17 — Held-out evaluation: all checkpoints x all seeds (BASE prompt throughout)
checkpoints = {
    "base": "base",
    "sft": "checkpoints/sft/round_3/final",
    "dpo": "checkpoints/dpo/final",
}
seeds = [42, 123, 456]

for ckpt_name, ckpt_path in checkpoints.items():
    for seed in seeds:
        # Attack prompts
        out = f"results/eval/{ckpt_name}/seed_{seed}/held_out.jsonl"
        print(f"\n{'='*60}")
        print(f"Running: {ckpt_name} seed={seed} (attacks)")
        !python colab/run_inference.py \
            --checkpoint {ckpt_path} --system-prompt base --seed {seed} \
            --input data/splits/held_out_eval.jsonl \
            --output {out}

        # Benign twins (for helpfulness judging)
        benign_out = f"results/eval/{ckpt_name}/seed_{seed}/benign_results.jsonl"
        print(f"Running: {ckpt_name} seed={seed} (benign)")
        !python colab/run_inference.py \
            --checkpoint {ckpt_path} --system-prompt base --seed {seed} \
            --input data/seeds/benign_twins.jsonl \
            --output {benign_out}

In [ ]:
# Cell 18 — Helpfulness judging (must run BEFORE metric evaluation)
# Judge benign results for all checkpoints x all seeds
import itertools
checkpoints = ["base", "sft", "dpo"]
seeds = [42, 123, 456]

for ckpt, seed in itertools.product(checkpoints, seeds):
    inp = f"results/eval/{ckpt}/seed_{seed}/benign_results.jsonl"
    print(f"Judging helpfulness: {ckpt} seed={seed}")
    !python scripts/04_evaluate.py --judge-helpfulness --input {inp}

In [ ]:
# Cell 19 — Full metric evaluation (ASR, HR, FRR, McNemar, Cohen's h)
# Run AFTER helpfulness judging so HR/FRR are accurate
!python scripts/04_evaluate.py --checkpoints base,sft,dpo --seeds 42,123,456

## Transfer + Detection + Figures + Upload

In [ ]:
# Cell 20 — Transfer evaluation (Mistral-7B, BASE prompt)
!python colab/run_transfer.py --input data/splits/held_out_eval.jsonl --seed 42
!python scripts/04_evaluate.py --judge-transfer --input results/transfer/mistral_results.jsonl

In [ ]:
# Cell 21 — Perplexity detection baseline
!python colab/run_perplexity.py --checkpoint base \
    --input data/splits/held_out_eval.jsonl \
    --english-input data/seeds/raw_seeds.jsonl \
    --output results/analysis/perplexity_results.json
!python scripts/08_detection_analysis.py

In [ ]:
# Cell 22 — Residual analysis + Figures + Cost report
!python scripts/04_evaluate.py --residual-analysis --checkpoint dpo
!python scripts/05_generate_figures.py --results-dir results/eval/
!python scripts/09_cost_report.py

In [ ]:
# Cell 23 — Upload to HuggingFace Hub
!python scripts/06_upload_hf.py

## ASR Dashboard — Re-run anytime to check all scores

In [ ]:
# Cell 24 — ASR Dashboard: all rounds at a glance
from medics.metrics import compute_asr, compute_per_strategy_asr
from medics.utils import load_jsonl
from pathlib import Path

print("=" * 60)
print("  MediCS ASR Dashboard")
print("=" * 60)

# Attack rounds
for r in range(1, 4):
    path = f"results/attacks/round_{r}/results.jsonl"
    if Path(path).exists():
        results = load_jsonl(path)
        asr = compute_asr(results)
        n_harmful = sum(1 for x in results if x.get("judge_label") == "harmful")
        strats = compute_per_strategy_asr(results)
        strat_str = " | ".join(f"{s}:{v:.0%}" for s, v in sorted(strats.items()))
        print(f"\n  Round {r}: ASR {asr:.1%} ({n_harmful}/{len(results)})")
        print(f"    {strat_str}")
    else:
        print(f"\n  Round {r}: not yet run")

# Held-out evaluation
print(f"\n{'='*60}")
print("  Held-Out Evaluation (seed=42)")
print("=" * 60)
for ckpt in ["base", "sft", "dpo"]:
    path = f"results/eval/{ckpt}/seed_42/held_out.jsonl"
    if Path(path).exists():
        results = load_jsonl(path)
        asr = compute_asr(results)
        print(f"  {ckpt:>5}: ASR {asr:.1%}")
    else:
        print(f"  {ckpt:>5}: not yet run")

# Transfer
transfer_path = "results/transfer/mistral_results.jsonl"
if Path(transfer_path).exists():
    results = load_jsonl(transfer_path)
    asr = compute_asr(results)
    print(f"\n  Transfer (Mistral-7B): ASR {asr:.1%}")